# Equational Theories Project (ETP) — Magma Exploration

The **Equational Theories Project (ETP)** is a collaborative mathematical research effort exploring the web of implications and non-implications between equational laws over **magmas** (sets equipped with a single binary operation $*$).

This notebook sets up and executes an **Attribute Exploration** over equational laws on magmas using Formal Concept Analysis (FCA). Through automated question generation and counterexample refutations (finite Cayley tables generated via constraint solving), we discover the minimal canonical implication basis describing the relationships between these algebraic laws.

## 1. Magmas, Terms, and Equations

A **Magma** $(M, *)$ is a set $M$ equipped with a binary operation $*: M \times M \to M$.

Terms are built from variables ($x, y, z, w, \dots$) and binary operations. An **Equation** $L = R$ is universally quantified over all variables occurring in $L$ and $R$.

In [ ]:
import sys
from pathlib import Path
REPO_ROOT = Path("..").resolve().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from explorations.equational_theories.magma import ETP, Equation, Magma, MagmaExpert, Op, Term, Var
from conceptual_exploration.core.theory import ImplicationTheory
from conceptual_exploration.exploration.base import ExplorationBase
from conceptual_exploration import AttributeExploration, Implication

# Parse terms and equations
comm = Equation.parse("x * y = y * x", name="Commutativity", id=43)
assoc = Equation.parse("(x * y) * z = x * (y * z)", name="Associativity", id=381)
idem = Equation.parse("x = x * x", name="Idempotence", id=3)

print(comm)
print(assoc)
print(idem)

## 2. Standard Finite Magmas and Cayley Tables

Finite magmas are represented by their Cayley multiplication tables. Let's inspect some standard finite magmas and verify which equational laws hold on them.

In [ ]:
# Cyclic addition Z_3(+)
z3 = Magma.cyclic_addition(3)
print(z3)
print("Commutative:", z3.holds(comm))
print("Associative:", z3.holds(assoc))
print("Idempotent: ", z3.holds(idem))

print("\n" + "="*40 + "\n")

# Rock-Paper-Scissors tournament magma
rps = Magma.rock_paper_scissors()
print(rps)
print("Commutative:", rps.holds(comm))
print("Associative:", rps.holds(assoc))
print("Idempotent: ", rps.holds(idem))

## 3. Selecting ETP Equational Laws

We select a representative set of equational laws from the ETP catalog and compute their **duality symmetry mappings** (the anti-automorphism $(x * y)^{\text{op}} = y^{\text{op}} * x^{\text{op}}$).

In [ ]:
equations = [
    Equation.parse("x = y", name="Singleton / Degenerate", id=2),
    Equation.parse("x = (x * x)", name="Idempotence", id=3),
    Equation.parse("x = (x * y)", name="Left-Zero / Left-Absorption", id=4),
    Equation.parse("x = (y * x)", name="Right-Zero / Right-Absorption", id=5),
    Equation.parse("x = ((x * y) * x)", name="Central-Identity", id=6),
    Equation.parse("x = (x * (y * x))", name="Left-Central-Identity", id=23),
    Equation.parse("((x * x) * y) = (x * (x * y))", name="Left-Alternative", id=7),
    Equation.parse("((y * x) * x) = (y * (x * x))", name="Right-Alternative", id=8),
    Equation.parse("((x * y) * x) = (x * (y * x))", name="Flexible", id=9),
    Equation.parse("(x * y) = (y * x)", name="Commutativity", id=43),
    Equation.parse("(x * y) = (x * (x * y))", name="Left-Idempotent-Composition", id=46),
    Equation.parse("((y * x) * x) = (y * x)", name="Right-Idempotent-Composition", id=47),
    Equation.parse("((x * y) * z) = (x * (y * z))", name="Associativity", id=381),
    Equation.parse("((x * y) * (z * w)) = ((x * z) * (y * w))", name="Medial / Entropic", id=4512),
    Equation.parse("(x * (x * y)) = y", name="Steiner Law 1", id=4687),
    Equation.parse("((y * x) * x) = y", name="Steiner Law 2", id=4688),
]

duality_map = ETP.get_duality_mapping(equations)
print(f"Loaded {len(equations)} equations closed under duality symmetry.")

## 4. Running Attribute Exploration

We initialize `ExplorationBase` with the equations and duality mappings, instantiate `MagmaExpert`, and run the `AttributeExploration` algorithm.

In [ ]:
base = ExplorationBase[Magma, Equation](
    attributes=equations,
    mappings=[duality_map],
)
expert = MagmaExpert(attributes=equations, max_search_size=3)
exploration = AttributeExploration(base, expert, evaluate_all=True)

state = exploration.run()

print("Exploration completed successfully!")
print(f"Questions Asked:        {state.questions_asked}")
print(f"Accepted Implications: {len(base.accepted_implications)}")
print(f"Total Theory Size:     {len(base.implications.implications)}")
print(f"Counterexamples Found: {len(state.counterexamples)}")

## 5. Discovered Implication Basis

Let's display the canonical simplified implication basis discovered for these equational laws.

In [ ]:
theory = ImplicationTheory(base.implications)
print("Discovered Canonical Implication Basis (Simplified):\n")
for idx, impl in enumerate(base.accepted_implications, start=1):
    simplified = theory.simplify(impl)
    premise_str = "{" + ", ".join(eq.name or str(eq) for eq in simplified.premise) + "}" if simplified.premise else "Ø"
    concl_str = "{" + ", ".join(eq.name or str(eq) for eq in simplified.conclusion) + "}"
    print(f"[{idx:02d}]  {premise_str}  ==>  {concl_str}")

## 6. Discovered Counterexample Magmas

Below are some of the finite counterexample magmas generated by the expert to refute invalid implications.

In [ ]:
unique_magmas = []
seen = set()
for cex in state.counterexamples:
    m = cex.object.original if hasattr(cex.object, "original") else cex.object
    key = (m.size, m.table)
    if key not in seen:
        seen.add(key)
        unique_magmas.append(m)

for i, m in enumerate(unique_magmas[:6], start=1):
    print(f"--- Counterexample Magma #{i} ---")
    print(m)
    print()